# Gene embedding comparison: Geneformer vs GenePT (text-embedding-3-large)

Compares pairwise cosine distances between genes in the two embedding spaces, on the monocyte HVG set (4032 genes). Picks gene groups that should be biologically close (S100A family, HLA class II, complement, interferon-response, etc.) and pairs that should be far (e.g. hemoglobin vs immune markers).

Embedding files: `monocytes_clean_geneformer.pt` (4032 × 1152), `monocytes_clean_genept_3large.pt` (4032 × 3072). Row order matches `monocytes_clean.h5ad` `var`.

In [ ]:
import numpy as np
import pandas as pd
import torch
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

NB = Path('/home/projects/nyosef/zvise/scvi-tools-neural-nmf/notebooks')
adata = ad.read_h5ad(NB / 'monocytes_clean.h5ad')
gene_names = adata.var['feature_name'].astype(str).to_numpy()
name_to_idx = {g: i for i, g in enumerate(gene_names)}

gf = torch.load(NB / 'monocytes_clean_geneformer.pt', map_location='cpu', weights_only=False).numpy()
gp = torch.load(NB / 'monocytes_clean_genept_3large.pt', map_location='cpu', weights_only=False).numpy()

def l2norm(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.where(n > 0, n, 1.0)

GF = l2norm(gf)
GP = l2norm(gp)
gp_zero = (np.linalg.norm(gp, axis=1) == 0)
print('geneformer:', GF.shape, 'genept:', GP.shape, 'genes:', len(gene_names))
print(f'GenePT zero-vector genes: {gp_zero.sum()} (no NCBI summary available)')

## Gene groups

Each group is a set of paralogs / functionally tight genes that *should* sit close together in a semantic embedding. Cross-group pairs should be far.

In [ ]:
GROUPS = {
    'S100A (calcium-binding, neutrophil/mono)': ['S100A8', 'S100A9', 'S100A12', 'S100A4', 'S100A6', 'S100A2'],
    'HLA class II (MHC-II)':                    ['HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DQA1', 'HLA-DQB1'],
    'Complement C1q':                           ['C1QA', 'C1QB', 'C1QC'],
    'Interferon-stimulated':                    ['IFIT1', 'IFIT2', 'IFIT3', 'IFI6', 'IFI44', 'IFI44L', 'IFI27'],
    'CXCL chemokines':                          ['CXCL1', 'CXCL2', 'CXCL3', 'CXCL8', 'CXCL9', 'CXCL10', 'CXCL11'],
    'Granzymes (cytotoxic)':                    ['GZMA', 'GZMB', 'GZMK'],
    'Hemoglobin':                               ['HBB', 'HBA1', 'HBA2'],
    'Mitochondrial-encoded':                    ['MT-ND1', 'MT-ND2', 'MT-ND3', 'MT-ND4', 'MT-ND5', 'MT-CO2', 'MT-CO3', 'MT-ATP6', 'MT-CYB'],
}

missing = {k: [g for g in v if g not in name_to_idx] for k, v in GROUPS.items()}
for k, v in missing.items():
    if v: print('  missing in', k, ':', v)
GROUPS = {k: [g for g in v if g in name_to_idx] for k, v in GROUPS.items()}
for k, v in GROUPS.items():
    print(f'{k:45s} n={len(v)}')

## Cell 1 — cosine distance for hand-picked pairs

Within-family pairs (should be small distance) vs cross-family pairs (should be large).

In [ ]:
def cos_dist(emb, a, b):
    return float(1.0 - emb[name_to_idx[a]] @ emb[name_to_idx[b]])

PAIRS = [
    ('close',  'S100A8',  'S100A9'),
    ('close',  'S100A8',  'S100A12'),
    ('close',  'HLA-DRA', 'HLA-DRB1'),
    ('close',  'HLA-DPA1','HLA-DPB1'),
    ('close',  'C1QA',    'C1QB'),
    ('close',  'C1QA',    'C1QC'),
    ('close',  'IFIT1',   'IFIT3'),
    ('close',  'IFI44',   'IFI44L'),
    ('close',  'CXCL9',   'CXCL10'),
    ('close',  'CXCL1',   'CXCL2'),
    ('close',  'GZMA',    'GZMB'),
    ('close',  'HBA1',    'HBA2'),
    ('close',  'HBA1',    'HBB'),
    ('close',  'MT-ND1',  'MT-ND2'),
    ('far',    'HBB',     'HLA-DRA'),
    ('far',    'HBB',     'S100A8'),
    ('far',    'HBB',     'CXCL10'),
    ('far',    'GZMB',    'C1QA'),
    ('far',    'GZMB',    'HBA1'),
    ('far',    'MT-CO2',  'HLA-DRA'),
    ('far',    'MT-ND1',  'CXCL10'),
    ('far',    'CD14',    'HBA1'),
    ('far',    'IFIT1',   'HBB'),
]

rows = []
for kind, a, b in PAIRS:
    if a in name_to_idx and b in name_to_idx:
        rows.append({
            'kind': kind, 'gene_A': a, 'gene_B': b,
            'geneformer_dist': cos_dist(GF, a, b),
            'genept_dist':     cos_dist(GP, a, b),
        })
pair_df = pd.DataFrame(rows)
pair_df

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for kind, color in [('close', 'tab:green'), ('far', 'tab:red')]:
    sub = pair_df[pair_df['kind'] == kind]
    ax.scatter(sub['geneformer_dist'], sub['genept_dist'], c=color, label=kind, s=40, alpha=0.8)
lim = (0, max(pair_df[['geneformer_dist', 'genept_dist']].max()) * 1.05)
ax.plot(lim, lim, 'k--', lw=0.5, alpha=0.4)
ax.set_xlabel('Geneformer cosine distance')
ax.set_ylabel('GenePT cosine distance')
ax.set_title('Close pairs cluster low, far pairs cluster high')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## Cell 2 — within-group vs between-group distance distributions

For each embedding space, summarize mean within-group distance vs mean between-group distance. A semantically useful embedding has *within < between*.

In [ ]:
def group_distance_summary(emb, groups):
    out = []
    for name, genes in groups.items():
        idx = np.array([name_to_idx[g] for g in genes])
        other = np.setdiff1d(np.arange(emb.shape[0]), idx)
        sims_in = emb[idx] @ emb[idx].T
        d_in = 1 - sims_in[np.triu_indices(len(idx), k=1)]
        d_out = 1 - (emb[idx] @ emb[other].T).ravel()
        out.append({'group': name, 'n': len(idx),
                    'within_mean': d_in.mean(), 'between_mean': d_out.mean(),
                    'gap': d_out.mean() - d_in.mean()})
    return pd.DataFrame(out)

summary_gf = group_distance_summary(GF, GROUPS).assign(space='geneformer')
summary_gp = group_distance_summary(GP, GROUPS).assign(space='genept')
summary = pd.concat([summary_gf, summary_gp], ignore_index=True)
summary.pivot(index='group', columns='space', values=['within_mean', 'between_mean', 'gap']).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, space in zip(axes, ['geneformer', 'genept']):
    sub = summary[summary['space'] == space].set_index('group')
    x = np.arange(len(sub))
    ax.bar(x - 0.18, sub['within_mean'],  width=0.36, label='within',  color='tab:green')
    ax.bar(x + 0.18, sub['between_mean'], width=0.36, label='between', color='tab:gray')
    ax.set_xticks(x)
    ax.set_xticklabels([g.split(' ')[0] for g in sub.index], rotation=40, ha='right')
    ax.set_title(f'{space}: within < between?')
    ax.set_ylabel('cosine distance')
    ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## Cell 3 — nearest neighbors of canonical genes

For a handful of well-known marker genes, list the top-10 nearest genes in each space. The neighborhoods should make biological sense — e.g. S100A8 should sit next to S100A9/S100A12; HLA-DRA next to other MHC-II.

In [ ]:
def top_neighbors(emb, gene, k=10):
    i = name_to_idx[gene]
    sims = emb @ emb[i]
    sims[i] = -np.inf
    top = np.argpartition(-sims, k)[:k]
    top = top[np.argsort(-sims[top])]
    return [(gene_names[j], 1 - float(sims[j])) for j in top]

QUERY = ['S100A8', 'HLA-DRA', 'C1QA', 'IFIT1', 'CXCL10', 'GZMB', 'HBB', 'MT-ND1', 'CD14']

for q in QUERY:
    if q not in name_to_idx:
        print(f'{q} missing'); continue
    nn_gf = top_neighbors(GF, q, 10)
    nn_gp = top_neighbors(GP, q, 10)
    df = pd.DataFrame({
        'geneformer_nn': [f'{g} ({d:.2f})' for g, d in nn_gf],
        'genept_nn':     [f'{g} ({d:.2f})' for g, d in nn_gp],
    })
    print(f'\n=== {q} ===')
    print(df.to_string(index=False))

## Cell 4 — group co-membership in top-k neighbors

Quantitative: for each gene in a group, what fraction of its top-k neighbors come from the same group? Higher = the embedding recovers the family.

In [ ]:
def neighbor_recall(emb, groups, k=10):
    rows = []
    for name, genes in groups.items():
        idx = np.array([name_to_idx[g] for g in genes])
        if len(idx) < 2: continue
        idx_set = set(idx.tolist())
        sims = emb[idx] @ emb.T
        for r, i in enumerate(idx):
            sims[r, i] = -np.inf
        top = np.argpartition(-sims, k, axis=1)[:, :k]
        hits = np.array([sum(1 for j in row if j in idx_set) for row in top])
        rows.append({'group': name, 'n': len(idx),
                     f'recall@{k}': hits.mean() / min(k, len(idx) - 1)})
    return pd.DataFrame(rows)

k = 10
rec = neighbor_recall(GF, GROUPS, k).rename(columns={f'recall@{k}': 'geneformer'}).merge(
      neighbor_recall(GP, GROUPS, k).rename(columns={f'recall@{k}': 'genept'}), on=['group', 'n'])
rec

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
x = np.arange(len(rec))
ax.bar(x - 0.18, rec['geneformer'], width=0.36, label='geneformer', color='steelblue')
ax.bar(x + 0.18, rec['genept'],     width=0.36, label='genept',     color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels([g.split(' ')[0] for g in rec['group']], rotation=40, ha='right')
ax.set_ylabel(f'recall@{k} (same-group fraction)')
ax.set_title('GenePT vs Geneformer recover gene families')
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## Cell 5 — distance heatmap on the union of all group genes

Block structure along the diagonal = embedding clusters genes by family. Off-diagonal blocks should be lighter (within) and darker (between).

In [ ]:
ordered_genes, group_labels = [], []
for name, genes in GROUPS.items():
    ordered_genes += genes
    group_labels += [name.split(' ')[0]] * len(genes)
idx = np.array([name_to_idx[g] for g in ordered_genes])

D_gf = 1 - GF[idx] @ GF[idx].T
D_gp = 1 - GP[idx] @ GP[idx].T

boundaries = np.cumsum([len(v) for v in GROUPS.values()])[:-1]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, D, title in zip(axes, [D_gf, D_gp], ['geneformer', 'genept']):
    im = ax.imshow(D, cmap='viridis', vmin=0, vmax=np.quantile(D, 0.98))
    for b in boundaries:
        ax.axhline(b - 0.5, color='white', lw=0.5)
        ax.axvline(b - 0.5, color='white', lw=0.5)
    ax.set_xticks(range(len(ordered_genes))); ax.set_xticklabels(ordered_genes, rotation=90, fontsize=6)
    ax.set_yticks(range(len(ordered_genes))); ax.set_yticklabels(ordered_genes, fontsize=6)
    ax.set_title(f'{title} cosine distance')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## Cell 6 — do the two spaces agree on a random sample of gene pairs?

Scatter pairwise distances from a random 5k-pair sample. Strong correlation = the spaces broadly agree on gene structure; weak = they encode different signal.

In [ ]:
rng = np.random.default_rng(0)
N = GF.shape[0]
n_pairs = 5000
i = rng.integers(0, N, n_pairs); j = rng.integers(0, N, n_pairs)
mask = i != j; i, j = i[mask], j[mask]
d_gf = 1 - (GF[i] * GF[j]).sum(1)
d_gp = 1 - (GP[i] * GP[j]).sum(1)
rho_p = np.corrcoef(d_gf, d_gp)[0, 1]
from scipy.stats import spearmanr
rho_s, _ = spearmanr(d_gf, d_gp)

fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(d_gf, d_gp, s=2, alpha=0.2)
ax.set_xlabel('geneformer cosine distance')
ax.set_ylabel('genept cosine distance')
ax.set_title(f'pearson r = {rho_p:.2f}, spearman ρ = {rho_s:.2f}  (n={len(i)})')
plt.tight_layout(); plt.show()